In [ ]:
_repo_root = !git rev-parse --show-toplevel
%cd {_repo_root[0]}
del _repo_root

# MovieLens scaling

Reader-facing companion for the MovieLens 20M repeated-shuffle scaling experiment. Its purpose is to test whether spectral neurons train and benefit from greater matrix dimension as the number of ratings processed grows. The stream automatically enters a fresh shuffle whenever the requested budget exceeds the training-pool size. FM is the parameter-matched capacity reference; this is not a recommender-system leaderboard study. Lower RMSE is better.

## Setup

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib as mpl
import numpy as np
import pandas as pd

from paper.experiments.movielens_scaling import (
    PROFILES,
    default_raw_path,
    select_lr,
    summarize_raw,
    validate_raw,
)
from paper.plotting import (
    plot_movielens_dimensions,
    plot_movielens_models_by_dimension,
    plot_movielens_warm_coverage,
)

mpl.rcParams["figure.dpi"] = 140

In [ ]:
profile_name = os.environ.get("MOVIELENS_SCALING_PROFILE", "full")
variant = os.environ.get("MOVIELENS_SCALING_VARIANT") or None
profile = PROFILES[profile_name]

raw_override = os.environ.get("MOVIELENS_SCALING_RAW_PATH")
raw_path = (
    Path(raw_override)
    if raw_override
    else default_raw_path(profile_name, variant)
)

## Context and assumptions

- The source is the [MovieLens 20M dataset](https://grouplens.org/datasets/movielens/20m/). The fixed split is random 80/10/10 within each user; one holdout rating is moved into training for any movie otherwise absent from the complete training pool.
- Users and movies are compact categorical IDs, so there is no fitted feature preprocessing. Ratings are shifted by the fixed midpoint 2.75 during training; RMSE is unchanged and remains in rating units.
- Each data seed defines a deterministic stream of fresh permutations over the complete training pool. A trajectory consumes one global minibatch stream across pass boundaries, so the x-axis is ratings processed by the optimizer—not unique ratings, independently trained datasets, or convergence-controlled fits.
- The explicit profile `train_sizes` are the only evaluation checkpoints and their maximum is the final budget. The full profile evaluates the power-of-two grid from $2^{20}$ through $2^{24}$ and stops there. The runner creates as many permutations as that budget requires; pass boundaries do not add checkpoints, and a minibatch may cross them.
- At every checkpoint, median validation RMSE across tuning seeds selects a learning rate separately for each model family and capacity. Evaluation starts from fresh initializations; checkpoints selecting the same rate share a trajectory, but a plotted line may stitch checkpoints from different trajectories when the selected rate changes. It is a validation-selected performance envelope, not one coherent training path.
- With one user and one movie field, FM is exactly biased matrix factorization. For spectral dimension $d$, both nonlinear families use $d(d+1)/2$ parameters per identity: one FM bias plus rank $d(d+1)/2-1$, versus one symmetric $d\times d$ matrix. Spectral uses the middle eigenvalue, $\lambda_{d // 2}$, and the profiles therefore use odd dimensions.
- Lines are medians across data-order and initialization seeds. Bands span the interquartile seed variation on one fixed test set; they are not confidence intervals for a test population.

## Load and validate results

The default basename includes `repeated_shuffle`, so legacy one-pass CSVs are neither loaded nor used as append targets. Set `MOVIELENS_SCALING_RAW_PATH` only to inspect an explicit compatible result file.

In [ ]:
if not raw_path.exists():
    raise FileNotFoundError(
        f"{raw_path} does not exist; "
        "run paper.experiments.movielens_scaling first"
    )

raw = pd.read_csv(raw_path)
validate_raw(raw, profile, variant)

## Validation selection and recorded capacity

In [ ]:
selected = select_lr(raw)
selected_lrs = (
    selected[
        [
            "train_size",
            "model",
            "dim",
            "rank",
            "parameters_per_identity",
            "selected_lr",
        ]
    ]
    .drop_duplicates()
    .sort_values(["train_size", "dim", "model"], kind="stable")
)
boundaries = selected_lrs.loc[
    np.isclose(selected_lrs["selected_lr"], min(profile.lrs))
    | np.isclose(selected_lrs["selected_lr"], max(profile.lrs))
]
if not boundaries.empty:
    warnings.warn(
        "selected learning rate touches the tuning-grid boundary for:\n"
        + boundaries.to_string(index=False),
        stacklevel=1,
    )

capacity_table = (
    selected[
        [
            "model",
            "dim",
            "rank",
            "parameters_per_identity",
            "num_parameters",
        ]
    ]
    .drop_duplicates()
    .sort_values(["dim", "model"], kind="stable")
    .reset_index(drop=True)
)
capacity_table

## Selected test summary

In [ ]:
summary = summarize_raw(raw)
summary[
    [
        "train_pool_size",
        "train_size",
        "model",
        "dim",
        "rank",
        "parameters_per_identity",
        "num_parameters",
        "selected_lr",
        "median_test_rmse",
        "q25_test_rmse",
        "q75_test_rmse",
        "median_test_warm_fraction",
        "n",
    ]
].sort_values(["train_size", "dim", "model"], kind="stable")

## Spectral neurons across dimensions

This is the central capacity-scaling view for the paper: it asks whether larger learned matrices become useful as more ratings are processed by the optimizer.

In [ ]:
if variant in (None, "spectral"):
    plot_movielens_dimensions(selected, "spectral")
else:
    print(
        "Spectral-dimension comparison requires the merged result "
        "or the spectral shard."
    )

## Matched model families within each dimension

Each facet fixes the spectral dimension and per-identity parameter budget. The same linear trajectory is repeated as a common reference.

In [ ]:
if variant is None:
    plot_movielens_models_by_dimension(selected)
else:
    print(
        "Matched-family facets require the merged result; "
        "set variant=None after appending all family shards."
    )

## FM across embedding ranks

The corresponding matrix-factorization view shows whether this dataset and training protocol expose a capacity effect in a familiar model family.

In [ ]:
if variant in (None, "fm"):
    plot_movielens_dimensions(selected, "fm")
else:
    print(
        "FM rank comparison requires the merged result or the FM shard."
    )

## Training-stream warm coverage

A validation or test identity is guaranteed to occur somewhere in the complete training pool, but it may not yet have occurred in the stream prefix consumed at an early checkpoint. This diagnostic shows how much of the fixed test set is warm at each checkpoint; it saturates at one after the first pass and remains there during later passes. Test RMSE itself is always computed on the complete fixed test set.

In [ ]:
plot_movielens_warm_coverage(selected);

## Reading the figures

The matched facets compare model families at a fixed per-identity capacity. The one-axis spectral plot is the primary test of whether the spectral family improves with dimension; the FM plot is a capacity-sensitive reference. The coverage plot should be read first when interpreting early checkpoints, because differences there can include behavior on identities not yet observed by the optimizer. At any requested checkpoint beyond one complete traversal of the training pool, coverage is fixed, so subsequent changes reflect additional optimization on reshuffled examples rather than newly warm identities.

Write substantive empirical conclusions here only after this notebook has executed against the complete result file.